# 02 · Referencia `average_all` (media de superficie esplérica)

**Rama `explore/topomap-refs`** · sobre los grises datos reales cacheados
(eegbci 64ch canónico, `standard_1005`, sujetos 1-2).

Concepto: en vez de promediar solo los 64 electrodos (CAR, muestreo discreto),
se estima el **campo sobre toda la malla** (`spherical_spline_matrix`) y se
toma la **media de superficie** como la referencia a eliminar:

\[ M_{avg\_all} = I - w\,1^\top, \qquad w_c = \frac{1}{N_p}\sum_{p\in
\mathrm{escalp}} M_{p,c} \]

con `M` la proyección píxel→canal del topomapa (`Σ_c w_c = 1`). Al haber ~1700
píxeles válidos frente a 64 canales, el promedio se acerca más a la *media
continua* del campo (limite físico del promedio de superficie), que ya es la
suposición del **REST** (referencia al infinito vía `leadfield`, `pinv`).

Validación (estadística robusta, las señales tienen artefactos):
1. anula la constante (`M_avg @ 1 = 0`, `rank = C-1`);
2. orden físico esperado en dispersión remanente: **car < avg_all < rest**
   (lo medido con mediana del MAD por canal: car ≈ 6.7, avg_all ≈ 6.8, rest ≫);
3. coherencia: `VE(avg_all, car)` alto (>0.95), `VE(avg_all, rest)` bajo
   (remplaza el modo común de canales, no el campo de dipolos globales).

Este operador será la **referencia de salida** del modelo del notebook 03
(10-10 bipolar → 10-20 monopolar con `average_all`).

## 0. Entorno y configuración

In [1]:
import os, sys, json
import numpy as np

ROOT = os.getcwd()
# ascenso hasta la raíz del repo (donde vive src/),
# robusto a la profundidad de notebooks/exploraciones/topomap_refs/
while not os.path.isdir(os.path.join(ROOT, "src")):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
for p in (os.path.join(ROOT, "src"),):
    if p not in sys.path:
        sys.path.insert(0, p)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import median_abs_deviation as mad

from eeg_transform.nb import load_experiment, multiconfig_data
from eeg_transform.references import build_reference_matrix
from eeg_transform.mapping import spherical_spline_matrix, scalp_grid_matrix

OUT = os.path.join(ROOT, "runs", "topomap_refs")
FIG = os.path.join(OUT, "02_figs")
os.makedirs(FIG, exist_ok=True)

def ve(a, b):
    a0 = a - a.mean(0, keepdims=True)
    b0 = b - b.mean(0, keepdims=True)
    return float(1 - ((a0 - b0) ** 2).sum() / (b0 ** 2).sum())

print("ROOT:", ROOT)

I0000 00:00:1790019103.561285  119959 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790019103.561633  119959 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790019103.596659  119959 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790019104.483884  119959 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790019104.484105  119959 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


ROOT: /home/aess/Proyectos/universal-eeg-transformer


## 1. Datos reales cacheados (canonical 64ch, refs)

In [2]:
cfg, ds = load_experiment("config/universal_refs.yaml")
mc = multiconfig_data(cfg, ds)["canonical"]
refs = mc.refs["test"]
C = ds.n_channels
Xu = refs["unipolar"]; Xcar = refs["car"]; Xr = refs["rest"]
Xb = refs["bipolar"]; Xle = refs["linked_ears"]; Xlm = refs["linked_mastoids"]
print("canales:", C, "| muestras test:", len(Xu))
print("referencias:", list(refs.keys()))

canales: 64 | muestras test: 3000
referencias: ['unipolar', 'linked_mastoids', 'linked_ears', 'bipolar', 'car', 'rest', 'laplacian']


## 2. Construcción de `average_all`

Se interpolan los 64 canales a la grilla de superficie (48×48 píxeles), se
toman solo los píxeles **válidos** (dentro del disco del cuero cabelludo) y se
promedia: `w = <M>_píxeles` (pesos de la media continua; `Σ_c w_c = 1`). La
referencia es `M_avg = I − w·1ᵀ` (convención por filas, `X_ref = X @ M_avg`).
Invariantes del operador:

* **anula la entrada constante por columnas**: `1ᵀ @ M_avg ≈ 0` (a un input
  con la misma señal en todos los canales —el modo común— le sale 0). Para
  `X @ M` el invariantete es una fila constante de `X`, i.e. `column` sums;
* **nulidad**: el vector `w` es el espacio nulo (`M_avg @ w ≈ 0`), así que
  `rank = C−1` hasta ruido numérico del spline (valor singular ~2e-8).

In [3]:
M, mask, xy = scalp_grid_matrix(ds.ch_positions, grid_px=48)
n_pix = int(mask.sum())
w = M[mask].mean(0)
M_avg = np.eye(C) - w[:, None]

colsum = np.abs(np.ones((1, C)) @ M_avg).max()
null_w = float(np.abs(M_avg @ w).max())
sv = np.linalg.svd(M_avg, compute_uv=False)[-1]
print("píxeles válidos:", n_pix, "| Σ w = %.9f" % w.sum())
print("|1ᵀ M_avg| max  = %.3e  (esperado ~0: anula el modo común)" % colsum)
print("|M_avg @ w| max  = %.3e  (nulidad = span(w) → rank = C-1)" % null_w)
print("menor valor singular M_avg = %.2e  → rank efectivo = %d (≈ C-1)"
      % (sv, int(np.linalg.matrix_rank(M_avg))))
assert colsum < 1e-6
Xavg = Xu @ M_avg

# figuras: campo de superficie (MAD por píxel) de car / avg_all y su diferencia
sf_car = mad((Xcar @ M.T)[:, mask], axis=0, scale=1.4826) * 1e6
sf_avg = mad((Xavg @ M.T)[:, mask], axis=0, scale=1.4826) * 1e6
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, fld, tt in zip(axes, [sf_car, sf_avg, np.abs(sf_avg - sf_car)],
                       ["car · MAD/píxel (µV)", "average_all · MAD/píxel (µV)", "|Δ MAD| (µV)"]):
    im = ax.scatter(xy[mask, 0], xy[mask, 1], c=fld, s=2, cmap="viridis")
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off"); ax.set_title(tt, fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("Construcción de average_all (media de superficie)", y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "avg_all_construccion.png"), dpi=110); plt.close(fig)

píxeles válidos: 1716 | Σ w = 1.000000000
|1ᵀ M_avg| max  = 2.678e-08  (esperado ~0: anula el modo común)
|M_avg @ w| max  = 1.184e-09  (nulidad = span(w) → rank = C-1)
menor valor singular M_avg = 2.17e-08  → rank efectivo = 64 (≈ C-1)


## 3. Validación física

* **Dispersión robusta** por canal (mediana del MAD): cuánto "mueve" la
  referencia; el orden esperado por física es **car < avg_all < rest** (la
  media de superficie anula el modo común global de forma más completa que el
  promedio de 64 canales, sin tocar el campo de dipolos).
* **VE** entre pares para situar el operador en el espacio de referencias:
  `avg_all` debe pegar a `car` (misma filosofía de promedio) y quedar lejos de
  `rest` (que además remueve contribuciones globales vía `leadfield`).
  Nota: `rest` en esta muestra destruida por deriva/artefacto lento.

In [4]:
def mad_uV(X): return float(np.median(mad(X, axis=0, scale=1.4826)) * 1e6)
disorder = {"unipolar": mad_uV(Xu), "car": mad_uV(Xcar), "avg_all": mad_uV(Xavg),
            "rest": mad_uV(Xr), "bipolar": mad_uV(Xb),
            "linked_mastoids": mad_uV(Xlm), "linked_ears": mad_uV(Xle)}
print("dispersión remanente (mediana MAD/canal, µV):")
for k, v in disorder.items():
    print("  %-15s %.2f" % (k, round(v, 2)))
order_ok = disorder["car"] < disorder["avg_all"] < disorder["rest"]
print("orden car < avg_all < rest:", order_ok)

v_ac = ve(Xavg, Xcar); v_ar = ve(Xavg, Xr); v_cr = ve(Xcar, Xr)
print(f"VE avg_all->car = {v_ac:.4f} | avg_all->rest = {v_ar:.4f} | car->rest = {v_cr:.4f}")

t = 200
fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.4))

def _topomap(Xt, positions, ax, title, grid_px=48):
    Mt, mt, xyt = scalp_grid_matrix(positions, grid_px=grid_px)
    img = (np.asarray(Xt).reshape(1, -1) @ Mt.T).reshape(grid_px, grid_px)
    img = np.where(mt.reshape(grid_px, grid_px), img, np.nan)
    vlim = float(np.nanmax(np.abs(img)))
    return ax.imshow(img, cmap="RdBu_r", vmin=-vlim, vmax=vlim, origin="lower", interpolation="bilinear")

for ax, k in zip(axes, ["car", "avg_all", "rest"]):
    X = refs[k] if k != "avg_all" else Xavg
    im = _topomap(X[t], ds.ch_positions, ax, f"{k} @ t={t}")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"{k} @ t={t}", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
fig.suptitle("average_all vs car vs rest (topomapa, misma muestra)", y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "avg_all_vs_refs.png"), dpi=110); plt.close(fig)
print("figuras ->", FIG)

dispersión remanente (mediana MAD/canal, µV):
  unipolar        9.54
  car             6.69
  avg_all         6.84
  rest            74.12
  bipolar         4.66
  linked_mastoids 10.01
  linked_ears     12.34
orden car < avg_all < rest: True
VE avg_all->car = 0.9630 | avg_all->rest = 0.0089 | car->rest = 0.0088


figuras -> /home/aess/Proyectos/universal-eeg-transformer/runs/topomap_refs/02_figs


## 4. Resumen y métricas

El operador `average_all` queda definido **dentro del notebook** (sin tocar
`src/`), como `M_avg = I - w·1ᵀ` con `w = <M>_píxeles` de la malla esférica.
Servirá de **referencia objetivo** del modelo en `03_modelo_montaje_referencia`.

Métricas guardadas en `runs/topomap_refs/02_metrics.json`.

In [5]:
results = {
    "n_pixeles_validos": n_pix,
    "sum_w": float(w.sum()),
    "anula_modo_comun": float(colsum),
    "nulidad_span_w": null_w,
    "valor_singular_min": float(sv),
    "VE_avg_all_car": v_ac, "VE_avg_all_rest": v_ar, "VE_car_rest": v_cr,
    "dispersion_mad_uV": disorder,
    "orden_car_lt_avg_all_lt_rest": bool(order_ok),
    "t": t,
}
with open(os.path.join(OUT, "02_metrics.json"), "w") as fh:
    json.dump(results, fh, indent=2, default=float)
print("métricas ->", os.path.join(OUT, "02_metrics.json"))
print("RESUMEN")
print("  orden físico car < avg_all < rest:", order_ok)
print("  VE(avg_all, car)=%.4f | VE(avg_all, rest)=%.4f" % (v_ac, v_ar))

métricas -> /home/aess/Proyectos/universal-eeg-transformer/runs/topomap_refs/02_metrics.json
RESUMEN
  orden físico car < avg_all < rest: True
  VE(avg_all, car)=0.9630 | VE(avg_all, rest)=0.0089
